# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagnik556/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis and time window

**Unit of analysis:** One row in the final analysis represents **one content item for one client** (`client_hash_id × content_hash_id`).

The source table is daily at the client × content × date level, so the daily rows will be aggregated within separate time windows.

**Feature window:** February 1, 2026 to February 28, 2026. These are the signals that would have been available at the decision point of February 28.

**Label window:** March 1, 2026 to March 31, 2026. This window is used only to define the later outcome.

The two windows do not overlap. This prevents March outcome information from leaking into February features.

In [26]:
%pip install -q duckdb huggingface_hub pandas

import os
import getpass
import duckdb
import pandas as pd

def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if token:
        return token

    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

token = get_hf_token()

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute("SET VARIABLE hf_token = ?", [token])
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM = f"{REL}/dim_content.parquet"

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

windows = con.sql(f"""
    SELECT
        'February features' AS window_name,
        MIN(report_date) AS first_day,
        MAX(report_date) AS last_day,
        COUNT(DISTINCT report_date) AS n_days,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_pages
    FROM {FEB}

    UNION ALL

    SELECT
        'March labels',
        MIN(report_date),
        MAX(report_date),
        COUNT(DISTINCT report_date),
        COUNT(DISTINCT client_hash_id),
        COUNT(DISTINCT content_hash_id)
    FROM {MAR}
""").df()

display(windows)

Paste your Hugging Face READ token (hf_...): ··········


,window_name,first_day,last_day,n_days,n_clients,n_pages
0,February features,2026-02-01,2026-02-28,28,54,321546
1,March labels,2026-03-01,2026-03-31,31,55,331437


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field contract

Every field is classified based on what would have been knowable at the February 28, 2026 decision point.

**Features — February 2026:**  
February performance features include total GSC impressions, total GSC clicks, CTR derived from clicks and impressions, impression-weighted average position, position variability, number of days with impressions, zero-click days with impressions, impression trend within February, and impression spikiness. Content-level features include content type, keyword token count, URL character count, category count, days since creation, search volume, competition, competition level, CPC, word count, and backlinks, with missing-value indicators where needed.

**Label — March 2026:**  
The target is `went_dark`, defined as a content item having zero measured GSC clicks during March 2026. March GSC clicks are used only to construct this later outcome and are never used as February features.

**Context:**  
`client_hash_id` and `content_hash_id` are used for joining and grouping. `report_date` and `month` define the time windows. `gsc_data_available` is used to distinguish measured data from unavailable data. `is_published` and `is_deleted` are used for filtering. Derived fields such as position tier and the three-way outcome are used for reporting or stratification, not as model inputs.

**Excluded:**  
`last_optimized_date`, `optimization_eligible_date`, and unsafe uses of `content_updated_date` are excluded because they can contain information from after the February 28 decision point. March GSC performance fields are excluded from features because they belong to the future label window. GA4/session/AI/scroll fields are excluded because they are downstream engagement signals and may also contain unmeasured zeros. Pseudonymous identifiers and internal tooling fields such as `keyword_hash_id`, `url_hash_id`, `provider_used`, and `model_used` are also excluded.

In [27]:
# Verify that the fields named in the contract exist in the warehouse.

fact_schema = con.sql(f"""
    DESCRIBE SELECT *
    FROM {FEB}
""").df()

dim_schema = con.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{DIM}')
""").df()

fact_fields = set(fact_schema["column_name"])
dim_fields = set(dim_schema["column_name"])

feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "ga4_pageviews",
    "ga4_sessions",
    "scroll_events",
    "content_type",
    "keyword_token_count",
    "url_char_count",
    "category_count",
    "content_created_date",
    "search_volume",
    "competition",
    "competition_level",
    "cpc",
    "word_count",
    "backlinks",
]

context_fields = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
    "gsc_data_available",
    "is_published",
    "is_deleted",
]

excluded_fields = [
    "keyword_hash_id",
    "url_hash_id",
    "provider_used",
    "model_used",
    "last_optimized_date",
    "optimization_eligible_date",
]

print("Feature fields present:")
print([c for c in feature_fields if c in fact_fields or c in dim_fields])

print("\nMissing from warehouse:")
print([c for c in feature_fields if c not in fact_fields and c not in dim_fields])

print("\nContext fields present:")
print([c for c in context_fields if c in fact_fields or c in dim_fields])

print("\nExcluded fields present:")
print([c for c in excluded_fields if c in fact_fields or c in dim_fields])

print("\nLabel:")
print("went_dark (derived from March 2026 GSC clicks)")

Feature fields present:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'ga4_pageviews', 'ga4_sessions', 'scroll_events', 'content_type', 'keyword_token_count', 'url_char_count', 'category_count', 'content_created_date', 'search_volume', 'competition', 'competition_level', 'cpc', 'word_count', 'backlinks']

Missing from warehouse:
[]

Context fields present:
['client_hash_id', 'content_hash_id', 'report_date', 'month', 'gsc_data_available', 'is_published', 'is_deleted']

Excluded fields present:
['keyword_hash_id', 'url_hash_id', 'provider_used', 'model_used', 'last_optimized_date', 'optimization_eligible_date']

Label:
went_dark (derived from March 2026 GSC clicks)


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [28]:
universe = con.sql(f"""
WITH feb_agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_feb,
        SUM(gsc_clicks) AS clk_feb
    FROM {FEB}
    WHERE gsc_data_available
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
       AND SUM(gsc_clicks) >= 3
)
SELECT f.*
FROM feb_agg f
JOIN read_parquet('{DIM}') d
    USING (client_hash_id, content_hash_id)
WHERE d.is_published
  AND d.content_created_date <= DATE '2026-02-28'
""").df()

print(f"Universe: {len(universe):,} content items")
print(f"Clients in universe: {universe.client_hash_id.nunique()}")

print("\nFebruary feature summary:")
display(universe[["imp_feb", "clk_feb"]].describe())

Universe: 29,700 content items
Clients in universe: 31

February feature summary:


,imp_feb,clk_feb
count,29700.000000,29700.000000
mean,4878.206195,18.595084
std,7771.160910,46.995007
min,100.000000,3.000000
25%,1322.750000,4.000000
50%,2561.000000,8.000000
75%,5370.000000,17.000000
max,167303.000000,3310.000000


In [29]:
# Build the March outcome separately from the February features.

label = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_mar,
        SUM(gsc_clicks) AS clk_mar
    FROM {MAR}
    WHERE gsc_data_available
    GROUP BY client_hash_id, content_hash_id
""").df()

frame = universe.merge(
    label,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

frame[["imp_mar", "clk_mar"]] = frame[["imp_mar", "clk_mar"]].fillna(0)

frame["went_dark"] = (frame["clk_mar"] == 0).astype(int)

print(f"Rows: {len(frame):,}")
print(f"Went-dark positives: {frame['went_dark'].sum():,}")
print(f"Base rate: {frame['went_dark'].mean():.3f}")

frame["outcome"] = "0_survived"
frame.loc[frame["imp_mar"] == 0, "outcome"] = "1_lost_all_visibility"
frame.loc[
    (frame["imp_mar"] > 0) & (frame["clk_mar"] == 0),
    "outcome"
] = "2_visible_but_zero_clicks"

print("\nOutcome mix:")
print(frame["outcome"].value_counts(normalize=True).sort_index().round(3))

Rows: 29,700
Went-dark positives: 1,506
Base rate: 0.051

Outcome mix:
outcome
0_survived                   0.949
1_lost_all_visibility        0.012
2_visible_but_zero_clicks    0.039
Name: proportion, dtype: float64


In [30]:
# Check missingness for the main February feature fields.

feature_missing = con.sql(f"""
WITH feb_features AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_feb,
        SUM(gsc_clicks) AS gsc_clicks_feb,
        SUM(gsc_sum_position) AS gsc_sum_position_feb,
        SUM(ga4_pageviews) AS ga4_pageviews_feb,
        SUM(ga4_sessions) AS ga4_sessions_feb,
        SUM(scroll_events) AS scroll_events_feb
    FROM {FEB}
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN gsc_impressions_feb IS NULL THEN 1 ELSE 0 END) AS missing_gsc_impressions,
    SUM(CASE WHEN gsc_clicks_feb IS NULL THEN 1 ELSE 0 END) AS missing_gsc_clicks,
    SUM(CASE WHEN gsc_sum_position_feb IS NULL THEN 1 ELSE 0 END) AS missing_gsc_position,
    SUM(CASE WHEN ga4_pageviews_feb IS NULL THEN 1 ELSE 0 END) AS missing_ga4_pageviews,
    SUM(CASE WHEN ga4_sessions_feb IS NULL THEN 1 ELSE 0 END) AS missing_ga4_sessions,
    SUM(CASE WHEN scroll_events_feb IS NULL THEN 1 ELSE 0 END) AS missing_scroll_events
FROM feb_features
""").df()

display(feature_missing)

,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_position,missing_ga4_pageviews,missing_ga4_sessions,missing_scroll_events
0,321546,0.0,0.0,0.0,149677.0,149677.0,149677.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

The analysis is limited to content items that meet the February eligibility rules: at least 100 impressions and 3 clicks, published by February 28, 2026, with usable GSC data. This means the final universe is not representative of every content item in the warehouse.

The March outcome is also limited to measured GSC data. A `went_dark` label means zero measured GSC clicks in March, not necessarily that the page received no real-world traffic. Missing GA4 and scroll data are common in the February window, so those fields should not automatically be treated as zero.

The feature window ends on February 28 and the label window begins on March 1, so future March performance is not used as a February feature. Content fields that may reflect changes after the decision point are excluded to reduce leakage risk.

The resulting dataset is therefore a decision-support dataset for prioritization, not proof that a page will lose traffic or that a particular intervention will improve performance.

In [31]:
# Verify that the feature and label windows are separated.

feature_max = con.sql(f"""
    SELECT MAX(report_date) AS feature_end
    FROM {FEB}
""").fetchone()[0]

label_min = con.sql(f"""
    SELECT MIN(report_date) AS label_start
    FROM {MAR}
""").fetchone()[0]

print(f"Feature window ends: {feature_max}")
print(f"Label window starts: {label_min}")
print(f"Windows separated: {feature_max < label_min}")

Feature window ends: 2026-02-28
Label window starts: 2026-03-01
Windows separated: True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.